### Why Causal Masking is Different from Normal Self-Attention

---

### Normal Self-Attention

In **standard self-attention**, every token can attend to **all other tokens** in the sequence.

Example sentence:

```
"The cat sat on the mat"
```

Attention matrix:

```
        T1  T2  T3  T4  T5  T6
T1      ✔   ✔   ✔   ✔   ✔   ✔
T2      ✔   ✔   ✔   ✔   ✔   ✔
T3      ✔   ✔   ✔   ✔   ✔   ✔
T4      ✔   ✔   ✔   ✔   ✔   ✔
T5      ✔   ✔   ✔   ✔   ✔   ✔
T6      ✔   ✔   ✔   ✔   ✔   ✔
```

Meaning:

```
every token can see every other token
```

This type of attention is used in:

* **BERT**
* **Vision Transformers**
* **Encoder models**

These models process the **entire sequence at once**, so they allow full attention between tokens.

---

### Causal Self-Attention

In **causal attention**, a token can only attend to:

```
previous tokens + itself
```

Example attention matrix:

```
        T1  T2  T3  T4  T5
T1      ✔   ✖   ✖   ✖   ✖
T2      ✔   ✔   ✖   ✖   ✖
T3      ✔   ✔   ✔   ✖   ✖
T4      ✔   ✔   ✔   ✔   ✖
T5      ✔   ✔   ✔   ✔   ✔
```

Future tokens are **masked**, meaning they cannot influence the current token.

---

### Why Causal Masking is Needed

Language models generate text **left → right**.

Example:

```
Input: "The cat sat on"
Target: "the"
```

If the model could see the next word:

```
"The cat sat on the"
```

then training would become **cheating**, because the model already knows the correct answer.

Causal masking ensures:

```
prediction(t) depends only on tokens ≤ t
```

---

### Advantages of Causal Attention

#### 1. Enables Autoregressive Generation

Models like GPT generate text **one token at a time**.

Example:

```
"The cat sat on"
        ↓
predict "the"
```

Without causal masking, the model would **look ahead and see the answer**.

---

#### 2. Prevents Information Leakage

Future tokens contain information that should **not be available during prediction**.

Masking ensures the model only uses **past context**.

---

#### 3. Aligns Training With Inference

During inference:

```
token1 → predict token2
token1 token2 → predict token3
```

Causal masking ensures that **training follows the same process**.

---

### Summary

| Type             | Attention Access     |
| ---------------- | -------------------- |
| Self-attention   | all tokens           |
| Causal attention | previous tokens only |

---


In [1]:
import torch

inputs = torch.tensor(
  [[0.43, 0.15, 0.89], # Your     (x^1)
   [0.55, 0.87, 0.66], # journey  (x^2)
   [0.57, 0.85, 0.64], # starts   (x^3)
   [0.22, 0.58, 0.33], # with     (x^4)
   [0.77, 0.25, 0.10], # one      (x^5)
   [0.05, 0.80, 0.55]] # step     (x^6)
)

In [ ]:
x = inputs[1] # second input element
d_in = inputs.shape[1] # the input embedding size, d=3
d_out = 2 # the output embedding size, d=2

In [6]:
from torch import nn
class SelfAttention_v2(nn.Module):

    def __init__(self, d_in, d_out, qkv_bias=False):
        super().__init__()
        self.W_query = nn.Linear(d_in, d_out, bias=qkv_bias)
        self.W_key   = nn.Linear(d_in, d_out, bias=qkv_bias)
        self.W_value = nn.Linear(d_in, d_out, bias=qkv_bias)

    def forward(self, x):
        keys = self.W_key(x)
        queries = self.W_query(x)
        values = self.W_value(x)
        
        attn_scores = queries @ keys.T
        attn_weights = torch.softmax(attn_scores / keys.shape[-1]**0.5, dim=-1)

        context_vec = attn_weights @ values
        return context_vec

torch.manual_seed(789)
sa_v2 = SelfAttention_v2(d_in, d_out)
print(sa_v2(inputs))

tensor([[-0.0739,  0.0713],
        [-0.0748,  0.0703],
        [-0.0749,  0.0702],
        [-0.0760,  0.0685],
        [-0.0763,  0.0679],
        [-0.0754,  0.0693]], grad_fn=<MmBackward0>)


In [18]:
queries = sa_v2.W_query(inputs)
keys = sa_v2.W_key(inputs) 
attn_scores = queries @ keys.T

attn_weights = torch.softmax(attn_scores / keys.shape[-1]**0.5, dim=-1)
print("attention weights : " ,attn_weights)
print('\n')
print("attention scores : " , attn_scores)

attention weights :  tensor([[0.1921, 0.1646, 0.1652, 0.1550, 0.1721, 0.1510],
        [0.2041, 0.1659, 0.1662, 0.1496, 0.1665, 0.1477],
        [0.2036, 0.1659, 0.1662, 0.1498, 0.1664, 0.1480],
        [0.1869, 0.1667, 0.1668, 0.1571, 0.1661, 0.1564],
        [0.1830, 0.1669, 0.1670, 0.1588, 0.1658, 0.1585],
        [0.1935, 0.1663, 0.1666, 0.1542, 0.1666, 0.1529]],
       grad_fn=<SoftmaxBackward0>)


attention scores :  tensor([[ 0.2899,  0.0716,  0.0760, -0.0138,  0.1344, -0.0511],
        [ 0.4656,  0.1723,  0.1751,  0.0259,  0.1771,  0.0085],
        [ 0.4594,  0.1703,  0.1731,  0.0259,  0.1745,  0.0090],
        [ 0.2642,  0.1024,  0.1036,  0.0186,  0.0973,  0.0122],
        [ 0.2183,  0.0874,  0.0882,  0.0177,  0.0786,  0.0144],
        [ 0.3408,  0.1270,  0.1290,  0.0198,  0.1290,  0.0078]],
       grad_fn=<MmBackward0>)


#### Sample method to mask the value by taking the shape of the attention_weight and we need to fill it with 1 or 0 to able to make the future token as unknown 


In [8]:
context_length = attn_scores.shape[0]
mask_simple = torch.tril(torch.ones(context_length, context_length))
print(mask_simple)

tensor([[1., 0., 0., 0., 0., 0.],
        [1., 1., 0., 0., 0., 0.],
        [1., 1., 1., 0., 0., 0.],
        [1., 1., 1., 1., 0., 0.],
        [1., 1., 1., 1., 1., 0.],
        [1., 1., 1., 1., 1., 1.]])


In [10]:
masked_simple = attn_weights*mask_simple
print(masked_simple)

tensor([[0.1921, 0.0000, 0.0000, 0.0000, 0.0000, 0.0000],
        [0.2041, 0.1659, 0.0000, 0.0000, 0.0000, 0.0000],
        [0.2036, 0.1659, 0.1662, 0.0000, 0.0000, 0.0000],
        [0.1869, 0.1667, 0.1668, 0.1571, 0.0000, 0.0000],
        [0.1830, 0.1669, 0.1670, 0.1588, 0.1658, 0.0000],
        [0.1935, 0.1663, 0.1666, 0.1542, 0.1666, 0.1529]],
       grad_fn=<MulBackward0>)


In [13]:
batch = torch.stack((inputs, inputs), dim=0)
print(batch.shape)

torch.Size([2, 6, 3])


### Why We Transpose Keys When Computing Attention Scores

The attention score formula is:

$$
QK^T
$$

In code:

```python
attn_scores = queries @ keys.transpose(1,2)
```

---

### Shapes of Matrices

Assume:

```
batch = B
tokens = T
embedding dimension = D
```

Queries:

```
Q → (B, T, D)
```

Keys:

```
K → (B, T, D)
```

---

### Goal

We want to compute a **similarity score between every pair of tokens**.

The output should be:

```
(B, T, T)
```

Meaning:

```
token1 similarity with token1..tokenT
token2 similarity with token1..tokenT
token3 similarity with token1..tokenT
```

Each token compares itself with **all other tokens**.

---

### Matrix Multiplication Rule

Matrix multiplication follows this rule:

```
(A × B)
```

Where:

```
A : (m, n)
B : (n, p)
```

The result becomes:

```
(m, p)
```

The **inner dimensions must match**.

---

### Without Transpose

If we try:

```
Q @ K
```

Shapes become:

```
(T, D) @ (T, D)
```

This is **invalid matrix multiplication** because the inner dimensions do not match.

---

### With Transpose

So we transpose the keys:

```
K^T → (D, T)
```

Now multiplication becomes:

```
(T, D) × (D, T)
```

Result:

```
(T, T)
```

This gives a **token-to-token similarity matrix**.

---

### Example

Suppose we have **3 tokens**.

Queries:

```
Q
token1 → q1
token2 → q2
token3 → q3
```

Keys:

```
K
token1 → k1
token2 → k2
token3 → k3
```

Attention score matrix:

```
q1·k1  q1·k2  q1·k3
q2·k1  q2·k2  q2·k3
q3·k1  q3·k2  q3·k3
```

Each value represents the **similarity between tokens**.

Example:

```
score(i,j) = similarity(query_i , key_j)
```

---

### Intuition

Think of it like this:

```
Query = question
Key   = index of information
```

Each query asks:

```
Which keys match me the most?
```

Transpose allows the model to compute **all token similarities in one matrix operation**.

---

### Key Idea

Attention computes:

```
similarity(query_i , key_j)
```

for **every pair of tokens**, producing the **attention matrix (T × T)**.

The transpose enables **efficient computation using matrix multiplication**.

---


In [ ]:
class CausalAttention(nn.Module):

    def __init__(self, d_in, d_out, context_length,
                 dropout, qkv_bias=False):
        super().__init__()
        self.d_out = d_out
        self.W_query = nn.Linear(d_in, d_out, bias=qkv_bias)
        self.W_key   = nn.Linear(d_in, d_out, bias=qkv_bias)
        self.W_value = nn.Linear(d_in, d_out, bias=qkv_bias)
        self.dropout = nn.Dropout(dropout) # dropout is applied during training not inference, so it does not affect the output during inference
        self.register_buffer('mask', torch.triu(torch.ones(context_length, context_length), diagonal=1)) # New

    def forward(self, x):
        b, num_tokens, d_in = x.shape # New batch dimension b
        # For inputs where `num_tokens` exceeds `context_length`, this will result in errors
        # in the mask creation further below.
        # In practice, this is not a problem since the LLM (chapters 4-7) ensures that inputs  
        # do not exceed `context_length` before reaching this forward method. 
        keys = self.W_key(x)
        queries = self.W_query(x)
        values = self.W_value(x)

        attn_scores = queries @ keys.transpose(1, 2) # Changed transpose
        attn_scores.masked_fill_(  # New, _ ops are in-place
            self.mask.bool()[:num_tokens, :num_tokens], -torch.inf)  # `:num_tokens` to account for cases where the number of tokens in the batch is smaller than the supported context_size
        attn_weights = torch.softmax(
            attn_scores / keys.shape[-1]**0.5, dim=-1
        )
        attn_weights = self.dropout(attn_weights) # New

        context_vec = attn_weights @ values
        return context_vec

torch.manual_seed(123)

context_length = batch.shape[1]
ca = CausalAttention(d_in, d_out, context_length, 0.0)

context_vecs = ca(batch)

print(context_vecs)
print("context_vecs.shape:", context_vecs.shape)

tensor([[[-0.4519,  0.2216],
         [-0.5874,  0.0058],
         [-0.6300, -0.0632],
         [-0.5675, -0.0843],
         [-0.5526, -0.0981],
         [-0.5299, -0.1081]],

        [[-0.4519,  0.2216],
         [-0.5874,  0.0058],
         [-0.6300, -0.0632],
         [-0.5675, -0.0843],
         [-0.5526, -0.0981],
         [-0.5299, -0.1081]]], grad_fn=<UnsafeViewBackward0>)
context_vecs.shape: torch.Size([2, 6, 2])


### Causal Self-Attention Implementation (PyTorch)

#### Overview

This code implements **causal self-attention**, which is used in **autoregressive language models** like GPT.

The key idea:

> Each token can attend only to **itself and previous tokens**, not future tokens.

This is achieved using a **causal mask (upper triangular mask)**.

The attention formula:

$$
Attention(Q,K,V) = softmax\left(\frac{QK^T}{\sqrt{d_k}} + mask\right)V
$$

Where the **mask blocks future tokens**.

---

### 1. Import and Class Definition

```python
class CausalAttention(nn.Module):
```

This defines a **custom PyTorch layer** implementing causal self-attention.

It inherits from:

```python
nn.Module
```

which is the base class for all neural network layers in PyTorch.

---

### 2. Constructor (`__init__`)

```python
def __init__(self, d_in, d_out, context_length, dropout, qkv_bias=False):
```

##### Parameters

| Parameter        | Meaning                        |
| ---------------- | ------------------------------ |
| `d_in`           | input embedding dimension      |
| `d_out`          | output embedding dimension     |
| `context_length` | maximum sequence length        |
| `dropout`        | dropout probability            |
| `qkv_bias`       | whether Q,K,V layers have bias |

Example:

```
d_in = 768
d_out = 768
context_length = 1024
```

---

### 3. Query, Key, Value Projection Layers

```python
self.W_query = nn.Linear(d_in, d_out, bias=qkv_bias)
self.W_key   = nn.Linear(d_in, d_out, bias=qkv_bias)
self.W_value = nn.Linear(d_in, d_out, bias=qkv_bias)
```

These layers project the input embeddings into:

```
Query vectors
Key vectors
Value vectors
```

Mathematically:

```
Q = XWq
K = XWk
V = XWv
```

##### Shapes

If input shape is:

```
(batch, tokens, d_in)
```

Then:

```
Q,K,V → (batch, tokens, d_out)
```

---

### 4. Dropout Layer

```python
self.dropout = nn.Dropout(dropout)
```

Dropout randomly **drops attention weights during training**.

Purpose:

```
prevent overfitting
```

Important:

```
Dropout works only during training
```

During inference it is **disabled automatically**.

---

### 5. Creating the Causal Mask

```python
self.register_buffer(
    'mask',
    torch.triu(torch.ones(context_length, context_length), diagonal=1)
)
```

This creates a **causal mask**.

Example mask (`context_length = 5`)

```
0 1 1 1 1
0 0 1 1 1
0 0 0 1 1
0 0 0 0 1
0 0 0 0 0
```

Meaning:

```
1 → future token
0 → allowed
```

`torch.triu` creates the **upper triangular matrix**.

`diagonal=1` keeps the diagonal as **allowed tokens**.

---

### 6. Why `register_buffer`

```python
self.register_buffer(...)
```

This tells PyTorch:

```
This tensor belongs to the model
but it is not trainable
```

Benefits:

* moves with model to GPU
* saved in checkpoints
* excluded from gradient updates

---

### 7. Forward Pass

```python
def forward(self, x):
```

Input shape:

```
x = (batch_size, num_tokens, d_in)
```

Example:

```
batch_size = 2
tokens = 5
embedding = 3
```

---

### 8. Extracting Dimensions

```python
b, num_tokens, d_in = x.shape
```

Meaning:

```
b = batch size
num_tokens = sequence length
d_in = embedding dimension
```

---

### 9. Compute Queries, Keys, Values

```python
keys = self.W_key(x)
queries = self.W_query(x)
values = self.W_value(x)
```

Shapes:

```
keys    → (b, tokens, d_out)
queries → (b, tokens, d_out)
values  → (b, tokens, d_out)
```

---

### 10. Attention Score Computation

```python
attn_scores = queries @ keys.transpose(1, 2)
```

Transpose:

```
keys: (b, tokens, d)
keys.transpose(1,2) → (b, d, tokens)
```

Matrix multiplication:

```
(b, tokens, d)
×
(b, d, tokens)
=
(b, tokens, tokens)
```

Result:

```
attention score matrix
```

---

### 11. Apply Causal Mask

```python
attn_scores.masked_fill_(
    self.mask.bool()[:num_tokens, :num_tokens],
    -torch.inf
)
```

Explanation:

```
future tokens → replaced with -∞
```

Because:

```
softmax(-inf) = 0
```

Masked tokens get **zero attention weight**.

---

### 12. Scaling Attention Scores

```python
attn_weights = torch.softmax(
    attn_scores / keys.shape[-1]**0.5,
    dim=-1
)
```

Scaling prevents **softmax saturation**.

Reason:

```
large dot products → unstable softmax
```

Scaling by √dₖ stabilizes training.

---

### 13. Applying Dropout

```python
attn_weights = self.dropout(attn_weights)
```

Dropout randomly removes some attention weights during training.

Example:

```
[0.3, 0.4, 0.3]
↓
[0.3, 0.0, 0.3]
```

---

### 14. Computing Context Vectors

```python
context_vec = attn_weights @ values
```

Shape:

```
(b, tokens, tokens)
×
(b, tokens, d_out)
=
(b, tokens, d_out)
```

Each token now contains information from **previous tokens**.

---

### 15. Return Output

```python
return context_vec
```

Output shape:

```
(batch_size, num_tokens, d_out)
```

---

### 16. Model Initialization

```python
torch.manual_seed(123)
```

Ensures **reproducible random results**.

---

### 17. Setting Context Length

```python
context_length = batch.shape[1]
```

Meaning:

```
maximum tokens supported by mask
```

---

### 18. Creating the Attention Layer

```python
ca = CausalAttention(d_in, d_out, context_length, 0.0)
```

Dropout:

```
0.0 → disabled
```

---

### 19. Running Attention

```python
context_vecs = ca(batch)
```

The batch embeddings are processed through **causal attention**.

---

### 20. Printing Output

```python
print(context_vecs)
print("context_vecs.shape:", context_vecs.shape)
```

Example output shape:

```
(batch_size, num_tokens, d_out)
```

Example:

```
(2, 5, 3)
```

---

### Attention Pipeline

```
Input Embeddings (B,T,D)
        │
        ▼
Linear Projections
(Q,K,V)
        │
        ▼
Q × Kᵀ
        │
        ▼
Scale by √d
        │
        ▼
Apply Causal Mask
        │
        ▼
Softmax
        │
        ▼
Dropout
        │
        ▼
Attention Weights × Values
        │
        ▼
Context Vectors
```

---

### Key Difference from SimpleAttention

| Feature        | SimpleAttention | CausalAttention |
| -------------- | --------------- | --------------- |
| batching       | ❌               | ✔               |
| masking        | ❌               | ✔               |
| dropout        | ❌               | ✔               |
| Linear layers  | ❌               | ✔               |
| GPT compatible | ❌               | ✔               |

---

✅ **Summary**

Causal attention ensures:

```
token t cannot attend to tokens > t
```

This property is essential for **autoregressive models like GPT**.

